# The Monday Effect -- a quantitative teardown
### Real total-return tape -- per-weekday HAC inference -- pre/post-2000 difference

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Monday Effect still there?: Busted](https://img.shields.io/badge/Monday_Effect_still_there%3F-Busted-8b949e?style=flat-square)


The deep companion to the [curious notebook](01_for_the_curious.ipynb).
We test three assertions: Monday is negative (false), a calendar rule beats
the index (mirage), and the effect may have decayed (not testable -- never
negative on this tape).

> Not investment advice. SPY daily, total-return; cash 0%; 1 bp/switch; no execution lag.
> Sources: docs/references.md. Run: docs/results.md.

**vs Study 90 (Weekend):** Study 90 used overnight returns; this study uses
close-to-close (French 1980 convention). Both find no negative Monday.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from monday_effect import data, strategy

AS_OF = "2026-06-16"
frame = data.load_real("SPY", mode="total_return").loc[:AS_OF]
close = frame["close"]
wm = strategy.weekday_means(close)
bh = strategy.buy_and_hold(close)
mon_only = strategy.backtest(close, strategy.monday_only_position(close), cost_bps=1.0)
skip_mon = strategy.backtest(close, strategy.skip_monday_position(close), cost_bps=1.0)
print(f"SPY total return: {len(close):,} rows  {close.index[0].date()} -> {close.index[-1].date()}  fingerprint={data.fingerprint(frame)}")
print(wm.round(2))

SPY total return: 8,401 rows  1993-01-29 -> 2026-06-16  fingerprint=b8fb5124747e
     mean_bps         n  hac_t
Mon     5.640 1,578.000  2.060
Tue     7.260 1,726.000  2.670
Wed     6.200 1,723.000  2.210
Thu     1.460 1,690.000  0.520
Fri     3.320 1,683.000  1.330


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | NONE | Monday-rest **+1.08 bps** (HAC *t* **+0.36**), wrong sign and insignificant. |
| **Tradability** | MIRAGE | Buy-Monday **1.34%** CAGR vs B&H **10.82%** (-9.5 pts/yr). |
| **Monday Effect still there?** | BUSTED | Monday pre-2000 +4.78 bps, post-2000 +0.09 bps; change HAC t -0.77. |

## 1 -- Protocol

Per-weekday mean with HAC Newey-West *t* (daily returns mildly autocorrelated) --
Monday-vs-rest contrast as difference of means with HAC SEs in quadrature --
pre-2000/post-2000 split with **test of the change** on the pre-registered millennium cut --
literal timers net of 1 bp/switch. **None trigger:** Monday contrast wrong sign and |t| << 2.

## 2 -- Per-weekday means and the Monday contrast

In [2]:
wm.round(3)

,mean_bps,n,hac_t
Mon,5.637,"1,578.000",2.061
Tue,7.257,"1,726.000",2.670
Wed,6.198,"1,723.000",2.209
Thu,1.457,"1,690.000",0.518
Fri,3.321,"1,683.000",1.326


In [3]:
mon = strategy.contrast(close, 0)
tue = strategy.contrast(close, 1)
print(f"Monday  - rest: {mon['diff_bps']:+.2f} bps   HAC t {mon['hac_t']:+.2f}")
print(f"Tuesday - rest: {tue['diff_bps']:+.2f} bps   HAC t {tue['hac_t']:+.2f}")
print('Monday is positive and non-significant: effect absent, not merely weak.')

Monday  - rest: +1.06 bps   HAC t +0.35
Tuesday - rest: +3.12 bps   HAC t +1.03
Monday is positive and non-significant: effect absent, not merely weak.


Monday-vs-rest: **+1.08 bps** (HAC t +0.36), **wrong sign**.
With five weekday tests in play, no contrast would survive a snooping correction.
The negative-Monday Effect is absent from this tape entirely.

## 3 -- Pre/post-2000 split with a test of the change

In [4]:
sp = strategy.subperiod_effect(close, 0, cut='2000-01-01')
print(f"Monday pre-2000:  {sp['pre_diff_bps']:+.2f} bps")
print(f"Monday post-2000: {sp['post_diff_bps']:+.2f} bps")
print(f"Change: {sp['change_bps']:+.2f} bps  HAC t(change): {sp['hac_t_change']:+.2f}")
print('Monday positive in BOTH halves. Cannot certify decay.')

Monday pre-2000:  +4.78 bps
Monday post-2000: +0.06 bps
Change: -4.72 bps  HAC t(change): -0.77
Monday positive in BOTH halves. Cannot certify decay.


The Monday Effect predates the SPY era. French found it on 1953-1977 data;
on the 1993-2026 tape Monday is positive in both sub-periods.
Change t = -0.77: not significant. We cannot certify decay -- nothing was there.

## 4 -- Literal timers net of 1 bp/switch

In [5]:
import pandas as pd
tbl = pd.DataFrame({
    'CAGR %':  [mon_only['cagr']*100, skip_mon['cagr']*100, bh['cagr']*100],
    'Vol %':   [mon_only['vol']*100, skip_mon['vol']*100, bh['vol']*100],
    'Sharpe':  [mon_only['sharpe'], skip_mon['sharpe'], bh['sharpe']],
    'MaxDD %': [mon_only['max_dd']*100, skip_mon['max_dd']*100, bh['max_dd']*100],
    'TiM %':   [mon_only['time_in_market']*100, skip_mon['time_in_market']*100, bh['time_in_market']*100],
}, index=['Buy Monday only', 'Skip Monday', 'Buy & hold'])
tbl.round(2)

,CAGR %,Vol %,Sharpe,MaxDD %,TiM %
Buy Monday only,1.340,8.810,0.200,-41.700,18.780
Skip Monday,7.350,16.360,0.520,-48.940,81.220
Buy & hold,10.870,18.580,0.650,-55.190,100.000


Lost equity premium on sat-out days swamps any weekday tilt.
Buy-Monday sits in cash 81% of the time and loses 9.5 pts/yr to buy-and-hold.

## 5 -- The verdict

Signal NONE (Monday-rest +1.08 bps, HAC t +0.36 -- wrong sign),
Tradability MIRAGE (buy-Monday -9.5 pts/yr, skip-Monday -3.5 pts/yr vs B&H),
Monday Effect still there? BUSTED (positive pre- and post-2000, change t -0.77).
French 1980 is not certifiable on the 1993-2026 SPY tape.

## 6 -- Going further

- Extend to **price-only ^GSPC** from 1950s to locate exactly when the effect vanished.
- Apply a **White Reality Check** over all five weekday rules for a snooping-aware p-value.
- **Conditional Monday:** Monday-after-a-down-Friday -- does the unconditional flat hide a tilt?